# Train or Adapt the Physical Plausibility Checker

Define what the checker should reject using your own binary labels: `1 = good/acceptable`, `0 = bad/reject`.
This notebook uses `checks/physical_plausibility`; its exported `.pt` bundle loads unchanged in the CLI and REST service.
Choose **scratch training**, **fine-tuning**, or **calibration only** below. Fine-tuning updates pretrained query and
classifier weights; calibration changes only a decision threshold.

Run the common setup, then **only one** of paths A, B or C, followed by evaluation and inference.
Do not use Run All: the three paths are alternatives.


## Architecture and resources

1. Sample 64 unique real frames across each video; reject shorter videos.
2. Letterbox to 384×384. Frozen V-JEPA 2.1 produces `[32,24,24,1024]` tokens.
3. Remove tokens entirely in padding; each remaining token's validity weight is the fraction of real image in its patch.
4. Normalize 1024-dimensional tokens. Scratch training fits statistics from weighted video means on **train only**;
   fine-tuning preserves source normalization.
5. Learn `attention = softmax(z @ query / sqrt(1024) + log(validity))`. The weighted sum is a 1024-dimensional vector;
   a linear head produces one **bad logit**. Class-balanced BCE plus query/head L2 learns 2,049 parameters.

Extraction requires CUDA and processes videos individually to preserve the pretrained token layout. Training batches
cached tokens on the GPU. Reserve roughly 21 MiB/video (up to 36 MiB/video) for FP16 caches. The source archive and
backbone checkpoint download into `TORCH_HOME` and are SHA-256 verified. The backbone never receives gradients.


## Environment and optional pretrained head

Clone this repository and set up a Python 3.13 CUDA environment with PyTorch, NumPy, OpenCV, einops, timm,
filelock and FFmpeg. Match the pinned versions in `requirements.txt`; ensure `ffmpeg`/`ffprobe` are on `PATH`.
Jupyter is only needed for the interactive notebook. From the repository root, in a Python 3.13 virtual environment:

```bash
python -m pip install torch==2.10.0 torchvision==0.25.0 --index-url https://download.pytorch.org/whl/cu128
python -m pip install jupyterlab numpy==2.4.6 filelock==3.32.2 einops==0.8.2 timm==1.0.28 \
  third_party/opencv/opencv_python_headless-4.13.0.92+noffmpeg.noipp-cp313-cp313-manylinux_2_28_x86_64.whl
python -m jupyter lab
```

Use the repository's FFmpeg runtime as described in its getting-started guide. Supported video codecs are H.264,
AV1 and VP9; H.264 decoding requires NVIDIA video-driver libraries. Match deployment's PyTorch/CUDA environment
when checking numerical parity.

For fine-tuning or calibration, use the public
[`nvidia/physical-plausibility-evaluator`](https://huggingface.co/nvidia/physical-plausibility-evaluator) FineVideo bundle.
Download it once in your environment (scratch training does not need it):

```bash
python -m pip install huggingface_hub
hf download nvidia/physical-plausibility-evaluator \
  finevideo-attention-linear-v1/model.pt \
  --revision a427d8a37cba9200df31190a4130e60c209acd41 --local-dir /models
sha256sum /models/finevideo-attention-linear-v1/model.pt
```

Expected SHA-256: `3e9c138560df5958f01c5f8629a3221f0d53e2c0459d013d353bc9fbe2c889e9`.
Set `SOURCE` below to this local file. Labels are quality1 = bad; quality2/useful and quality3/artifact-free = good.
Validate this label definition against your task before adaptation. The backbone is downloaded separately.


In [ ]:
import logging
from pathlib import Path
import sys

import torch

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "checks/physical_plausibility/model.py").is_file()
)
sys.path.insert(0, str(ROOT))

from checks.physical_plausibility.attention import load_bundle, probability, threshold_logit
from checks.physical_plausibility.evaluation import calibrate, evaluate
from checks.physical_plausibility.model import VjepaTokenEncoder, file_sha256
from checks.physical_plausibility.tokens import atomic_json, extract, inventory, training_roles
from checks.physical_plausibility.training import settings, train

logging.basicConfig(level=logging.INFO)
assert torch.cuda.is_available(), "CUDA is required for extraction"


## Prepare manifests

JSONL has one local video per row. Relative video paths resolve relative to the manifest. Use `group_id` for related
clips (same scene/source/camera or clean/bug variants), and keep groups disjoint. Keep the natural validation/test
balance; the class-balanced training loss avoids discarding bad videos.

Training manifest, with **both labels in both splits** (add more rows than this format example):
```json
{"sample_id":"a-1","video_path":"videos/a.mp4","label":1,"group_id":"scene-a","split":"train"}
{"sample_id":"b-1","video_path":"videos/b.mp4","label":0,"group_id":"scene-b","split":"val"}
```
Use a separate test manifest with the same fields and `split: "test"`. Calibration-only adaptation uses a separate
manifest with `split: "calibration"`. A 60/20/20 group split is a reasonable training starting point.
Both classes are required for evaluation and calibration. Never choose settings or thresholds on the test set.


In [ ]:
TEST_MANIFEST = Path("/data/test.jsonl")
OUTPUT = Path("/results/my-checker")
DEVICE = "cuda"
CAP = 0.3  # Validation good-rejection cap, not a probability threshold.
LABEL_DEFINITION = "1 = acceptable for my task; 0 = reject for my task"
assert 0 <= CAP < 1, "CAP must be in [0, 1)"
OUTPUT.mkdir(parents=True, exist_ok=True)
CHECKPOINT = None  # Set by the one path you execute below.


## Common validation and extraction

The helper below checks splits before extraction. FP16 caches are atomic, resumable and bound to video hashes and the
token contract; changed inputs require a new cache directory. Computation uses Torch; caches store NumPy arrays.

Within your dataset, related clips must share a `group_id` and stay in one split. Against a pretrained checker,
we can detect identical source-training videos by file hash, but not related scenes or re-encoded versions with
different hashes. Check the source dataset provenance yourself when constructing your held-out data.


In [ ]:
test_rows = inventory(TEST_MANIFEST)
assert {r["split"] for r in test_rows} == {"test"}
assert {r["label"] for r in test_rows} == {0, 1}


def prepare_tokens(manifest, cache, *, source=None, calibration=False):
    rows = inventory(manifest)
    if calibration:
        assert {r["split"] for r in rows} == {"calibration"}
        assert {r["label"] for r in rows} == {0, 1}
    else:
        training_roles(rows, "group_id")
    for field in ("sample_id", "video_sha256", "group_id"):
        assert all(r.get(field) for r in rows + test_rows), f"Missing {field}"
        assert not ({r[field] for r in rows} & {r[field] for r in test_rows}), f"Test overlap: {field}"
    if source is not None:
        _, source_bundle = load_bundle(source)
        source_seen = set(source_bundle["training_and_selection_video_sha256"])
        held_out = test_rows + [r for r in rows if r["split"] != "train"]
        assert not source_seen & {r["video_sha256"] for r in held_out}, "Source training/selection overlap"
        logging.info("Source SHA-256: %s", file_sha256(source))
    return extract(manifest, cache, device=DEVICE)


## Path A — Train from scratch

Run these two cells only if you want a new query and linear classifier. Normalization is fitted on training videos only.
Adam starts at LR 0.003 with L2 0.001 for the query and classifier. Defaults run at least 3,600 updates and 10 epochs;
validation AUROC selects the checkpoint (including step zero), then validation chooses each requested threshold.
Resume files preserve optimizer state and reject changed data/settings. These are starting settings, not universal optima.


In [ ]:
FIT_MANIFEST = Path("/data/train-val.jsonl")
CONFIG = settings(caps=sorted({0.1, 0.2, CAP}))
# Optional: lr, head_l2, query_l2, batch_size, or optimizer="sgd" (momentum 0.9).


In [ ]:
fit_store = prepare_tokens(FIT_MANIFEST, OUTPUT / "fit-tokens")
CHECKPOINT = OUTPUT / "scratch.pt"
bundle = train(fit_store, CHECKPOINT, config=CONFIG, device=DEVICE,
               group_field="group_id", label_definition=LABEL_DEFINITION)


## Path B — Fine-tune a pretrained FineVideo checker

Run these cells instead of A or C. Set `SOURCE` to your verified local FineVideo attention bundle.
This restores the saved query and classifier and continues training them; it does not reset the classifier.
Source normalization and V-JEPA remain frozen. Default LR is 0.0001; validation selection and resume work as in path A.


In [ ]:
SOURCE = Path("/models/finevideo-attention-linear-v1/model.pt")
FIT_MANIFEST = Path("/data/train-val.jsonl")
CONFIG = settings(finetune=True, caps=sorted({0.1, 0.2, CAP}), scope="all")
# scope="linear" updates only the classifier; scope="all" also updates the query.


In [ ]:
fit_store = prepare_tokens(FIT_MANIFEST, OUTPUT / "fit-tokens", source=SOURCE)
CHECKPOINT = OUTPUT / "finetuned.pt"
bundle = train(fit_store, CHECKPOINT, source=SOURCE, config=CONFIG, device=DEVICE,
               group_field="group_id", label_definition=LABEL_DEFINITION)


## Path C — Calibrate a pretrained FineVideo checker

Run these cells instead of A or B when you only want to adjust the rejection threshold. No optimizer runs;
all model weights remain unchanged. Use a labeled calibration split separate from test data and source training/selection.
Only the requested cap is updated. Other saved thresholds remain available but are **not recalibrated** on your data.


In [ ]:
SOURCE = Path("/models/finevideo-attention-linear-v1/model.pt")
CALIBRATION_MANIFEST = Path("/data/calibration.jsonl")


In [ ]:
calibration_store = prepare_tokens(CALIBRATION_MANIFEST, OUTPUT / "calibration-tokens",
                                   source=SOURCE, calibration=True)
CHECKPOINT = OUTPUT / "calibrated.pt"
bundle = calibrate(calibration_store, SOURCE, CHECKPOINT, cap=CAP, device=DEVICE)


## Evaluate on untouched test videos

Bad recall is the fraction of bad clips caught; good rejection is the fraction of acceptable clips rejected.
A validation cap is **not a guarantee** on unseen data. AUROC measures ranking across thresholds. Save reports to your
experiment directory, not the repository; never calibrate on test data.


In [ ]:
assert CHECKPOINT is not None, "Run one of paths A, B or C first"
test_store = extract(TEST_MANIFEST, OUTPUT / "test-tokens", device=DEVICE)
report = evaluate(test_store, CHECKPOINT, cap=CAP, device=DEVICE)
atomic_json(OUTPUT / "test-metrics.json", report)
report


## Fresh inference

Saved decisions compare logits to avoid sigmoid saturation. `bad_score` is a ranking score, not an assurance of
calibrated probability. The following uses the same encoder and head as the service.


In [ ]:
VIDEO = Path("/data/new-video.mp4")
head, bundle = load_bundle(CHECKPOINT, device=DEVICE)
encoder = VjepaTokenEncoder(DEVICE)
with torch.inference_mode():
    tokens, weights = encoder.encode(VIDEO)
    logit = float(head(tokens[None], weights[None])[0])
cutoff = threshold_logit(bundle, CAP)
{"bad_score": float(probability(logit)), "prediction": "bad" if logit >= cutoff else "good"}


## Deploy the exported bundle

From the repository root, using a model path accessible inside Dazel:
```bash
source tools/env_setup.sh
dazel run //checks/physical_plausibility:run -- --video-path /data/new-video.mp4 --checkpoint-uri /models/checker.pt --output-dir /results
# For the REST service, set PHYSICAL_PLAUSIBILITY_CHECKPOINT_URI in ~/.cosmos_evaluator/.env.
dazel run //services/physical_plausibility:rest_api
```
Set `cap` in `metropolis.physical_plausibility` to a threshold in the bundle (default `0.3`). CLI/service
`bad_threshold` overrides use **scores in [0,1]**, not rejection caps. Use `--config-dir` for custom CLI config or
`PHYSICAL_PLAUSIBILITY_CONFIG_DIR` for the service. Mount model/config paths when running in a container.
See the [checker guide](../checks/physical_plausibility.md) for service configuration.
